In [1]:
import pandas as pd
from functions.eval import *
from transformers import pipeline, AutoTokenizer
import torch
from tqdm.notebook import tqdm
tqdm.pandas()

In [2]:
model_pred_col = "Camelbert-MSA"
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa-sentiment"
# model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"
# model_pred_col = "AraBert"

In [3]:
xai_exec = pd.read_csv("data/xai_exec/xai_exec_" + model_pred_col + ".csv")

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_name)
pipe = pipeline("text-classification", model=model_name, top_k=None, device=device, 
                truncation=True, max_length=512)

Device set to use cuda


In [5]:
xai_exec.columns

Index(['text', 'Camelbert-MSA', 'SHAP', 'LIME', 'IG', 'DeepLIFT',
       'EnsembleXAI_LIME_SHAP_IG_DL_mean',
       'EnsembleXAI_LIME_SHAP_IG_DL_median', 'EnsembleXAI_LIME_SHAP_IG_mean',
       'EnsembleXAI_LIME_SHAP_IG_median', 'EnsembleXAI_LIME_SHAP_mean',
       'EnsembleXAI_LIME_SHAP_median'],
      dtype='str')

In [6]:
ensemble_eval = xai_exec[["text", model_pred_col, "EnsembleXAI_LIME_SHAP_mean"]]

In [7]:
prediction_cache = {}

In [ ]:
hard_rationale_choices = {
    "elbow": {
        "elbow_method": ["simple-lmethod", "kneedle", "dfdt", "lmethod"],
    },
    "top_n": {
        "n": [3, 5, 10, 20],
    },
    "threshold": {
        "k": [0.1, 0.3, 0.5, 0.7],
    }
}

In [ ]:
for rationale_type in hard_rationale_choices.keys():
    for choice, values in hard_rationale_choices[rationale_type].items():
        for value in values:
            ensemble_eval["EXAI_LIME_SHAP_mean_" + rationale_type + "_" + str(value) + "_comprehensiveness"] = \
                ensemble_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], 
                                                            eval(row["EnsembleXAI_LIME_SHAP_mean"]), 
                                                            class_proba(pipe, row["text"]), prediction_cache,
                                                            method=rationale_type, **{choice: value}), axis=1)
            ensemble_eval["EXAI_LIME_SHAP_mean_" + rationale_type + "_" + str(value) + "_sufficiency"] = \
                ensemble_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], 
                                                            eval(row["EnsembleXAI_LIME_SHAP_mean"]), 
                                                            class_proba(pipe, row["text"]), prediction_cache,
                                                            method=rationale_type, **{choice: value}), axis=1)
            ensemble_eval["EXAI_LIME_SHAP_mean_" + rationale_type + "_" + str(value) + "_combined"] = \
                ensemble_eval.apply(lambda row: combined_metric(row["EXAI_LIME_SHAP_mean_" + rationale_type + "_" + str(value) + "_comprehensiveness"], 
                                                            row["EXAI_LIME_SHAP_mean_" + rationale_type + "_" + str(value) + "_sufficiency"], None, None, None), axis=1)

In [10]:
ensemble_eval.to_csv("data/hard_rationale/hard_rationale_ensemble_" + model_pred_col + ".csv", index=False)